# Chapter 3 - Lab 4: <font color='blue'>AutoGen Agent</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using Microsoft AutoGen (v0.4+)** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

AutoGen reframes agents as **participants in a conversation**. Even a single-agent task is modelled as a team of one running in a round-robin chat — a design choice that makes extension to multi-agent scenarios trivial.

In this lab you will wrap our two shared tools in AutoGen's `FunctionTool`, instantiate an `AssistantAgent`, and run it inside a `RoundRobinGroupChat` with a message-count termination.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **AutoGen v0.4+** (`autogen-agentchat`, `autogen-ext`, `autogen-core`) — modular agent framework.
* **OpenAI** `gpt-4o-mini` (via `OpenAIChatCompletionClient`).
* **Python `asyncio`** — AutoGen v0.4+ is async-first.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q autogen-agentchat autogen-ext autogen-core pydantic python-dotenv

## 2. Set up the API key(s)

This lab needs the following key(s):

  * **`OPENAI_API_KEY`** — your OpenAI key

If you are running this notebook in **Google Colab**, add each key in the left vertical menu under the *key* icon, using the names above.

If you are running locally, set the same names as environment variables (or load them from a `.env` file).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume env vars are already set (e.g. via .env).
    pass

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Wrap shared tools in `FunctionTool`

AutoGen's `FunctionTool` adapts a plain Python function into something an agent can call. The `description` is what the LLM sees when deciding whether to call this tool.

In [ ]:
from autogen_core.tools import FunctionTool

get_stock_data_tool = FunctionTool(
    get_stock_data,
    description='Get stock price and EPS data for a ticker symbol (e.g., AAPL, JPM)',
)
compute_pe_tool = FunctionTool(
    compute_pe,
    description='Compute P/E ratio given price and EPS',
)

## 5. Create the model client and the agent

AutoGen separates the **model client** (`OpenAIChatCompletionClient`) from the **agent** (`AssistantAgent`). This decoupling lets you swap models — or even mix models in a single team — without touching agent logic.

In [ ]:
import os
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    api_key=os.environ['OPENAI_API_KEY'],
)

analyst_agent = AssistantAgent(
    name='FinancialAnalyst',
    model_client=model_client,
    tools=[get_stock_data_tool, compute_pe_tool],
    system_message=system_message,
)

## 6. Run inside a `RoundRobinGroupChat`

Even with one agent we use a team — that's how AutoGen exposes multi-turn tool calling. The `MaxMessageTermination` caps the conversation so a runaway loop cannot drain your credits.

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

team = RoundRobinGroupChat(
    participants=[analyst_agent],
    termination_condition=MaxMessageTermination(10),
)

result = await team.run(task=input_message)

## 7. Inspect the conversation

`result.messages` contains every message exchanged — including tool calls and their results. Walk it to see what the agent did.

In [ ]:
for msg in result.messages:
    source = getattr(msg, 'source', '?')
    content = getattr(msg, 'content', '')
    print(f'[{source}] {str(content)[:200]}')

## 8. Results

You should see the agent call `get_stock_data` (once per ticker), then call `compute_pe` to compute each P/E ratio, and finally produce a short memo comparing the two.

**What to notice about AutoGen specifically:**

* The **chat-as-execution-model** generalises beautifully: scaling from 1 agent to N is a matter of adding `participants`, not rewriting the runtime.
* Termination conditions are first-class — you decide upfront how long a conversation may run.
* The model client / agent split is one of AutoGen's strongest design choices for mixed-model teams.
* Trade-off: even simple tasks have noticeable ceremony (imports, async, team-of-one). The pay-off appears once you start composing real multi-agent systems.